# Лабораторная работа 2. Текстовое ранжирование: TF-IDF и BM25 на WikiQA
## Шаблон студента

**ФИО:** _укажите_
**Группа:** _укажите_
**Вариант:** _укажите номер (1–25)_

---

**Цель работы:** освоить построение и сравнение классических алгоритмов текстового
ранжирования TF-IDF и BM25 на англоязычном корпусе, научиться вычислять DCG/NDCG
и интерпретировать причины различий поисковой выдачи.

**Постановка задачи.** Для английского вопроса требуется расположить предложения
связанной статьи Wikipedia так, чтобы предложения, содержащие правильный ответ,
оказались как можно выше в выдаче.

**Как работать с ноутбуком**

1. Ячейки с пометкой `# ВЫДАНО` менять не нужно — это инфраструктура.
2. Ячейки с `# TODO` содержат задания. Каждое задание закрывается автотестом
   (`test_*`): пока задание не выполнено, тест падает.
3. В конце есть ячейка «Полная самопроверка» — перед сдачей она должна пройти целиком.
4. Перед сдачей выполните **Kernel → Restart & Run All**: ноутбук обязан
   воспроизводиться сверху вниз без ручных правок.

**Что нельзя делать**

* использовать колонку `Label` при вычислении оценок ранжирования (это утечка
  разметки; автотест её обнаруживает);
* строить индекс только по кандидатам одного вопроса — IDF и средняя длина документа
  считаются по всему выбранному split;
* использовать разную токенизацию для TF-IDF и BM25.

## 0. Зачем это нужно и как устроен пайплайн

### Какая задача решается

Дан короткий запрос и набор текстов-кандидатов — нужно **упорядочить кандидатов по
убыванию полезности для запроса**. Поисковая система ничего не «понимает»: она
считает числовую оценку для каждой пары «запрос — документ» и сортирует.

### Где это применяется на практике

| Где встречается | Что делает ранжирование |
|---|---|
| Поиск по материалам курса в LMS (Moodle, Canvas) | упорядочивает лекции, конспекты и задания по запросу студента |
| Подбор текстов и заданий по теме | ранжирует фрагменты банка заданий под формулировку учителя |
| Чат-бот поддержки / FAQ организации | выбирает предложения-кандидаты из регламентов и методичек |
| Антиплагиат | ищет наиболее похожие фрагменты в коллекции |
| Проверка открытых ответов по эталону | сопоставляет ответ обучающегося с эталонными формулировками |
| RAG-системы поверх больших языковых моделей | отбирает контекст, который подаётся в модель |

Важно: **первый этап почти всех современных систем остаётся лексическим**. BM25
работает как быстрый отборщик кандидатов, а тяжёлая нейросетевая модель
переранжирует только верхушку списка. Поэтому TF-IDF и BM25 не устарели.

### Минимальная теория в шести пунктах

1. **Мешок слов** — текст превращается в набор токенов, порядок слов теряется.
2. **TF** — сколько раз терм встречается в документе («документ про это»).
3. **IDF** — насколько терм редкий по корпусу («терм различает документы»).
4. **TF-IDF + косинус** — вес равен произведению TF на IDF, сходство считается как
   косинус угла между векторами.
5. **BM25** — то же плюс две поправки: вклад повторов **насыщается** ($k_1$), длина
   документа штрафуется **явно** ($b$).
6. **DCG/NDCG** — метрика порядка: чем ниже позиция релевантного документа, тем
   меньше его вклад; NDCG нормирует результат на идеальную выдачу.

### Пайплайн работы

```
  WikiQA (633 вопроса, 6165 предложений, Label 0/1)
        v
  [1] Токенизация           единая для запросов и документов
        v
  [2] Индекс по корпусу     IDF и avgdl считаются по ВСЕМ предложениям split
        v
  [3] Скоринг               TF-IDF | BM25(k1, b) | наивные ориентиры
        v
  [4] Сортировка            внутри candidate set конкретного вопроса
        v
  [5] Метрика               DCG@k, NDCG@k; вопросы без Label=1 исключаются
        v
  [6] Анализ ошибок         запросы с максимальным |ΔNDCG|, языковой разбор
        v
  [7] Масштабирование       выгрузка оценок в BigQuery, пересчёт NDCG на SQL
```

| Шаг | Что ломается, если сделать неправильно |
|---|---|
| [1] единая токенизация | разница методов объяснится предобработкой, а не формулами |
| [2] индекс по всему split | на 9 предложениях статистика IDF и `avgdl` шумная |
| [3] наивные ориентиры | непонятно, много ли «NDCG = 0.64» или мало |
| [4] сортировка внутри candidate set | метрика начнёт мерить отбор, а не ранжирование |
| [5] исключение вопросов без ответа | NDCG превратится в долю вопросов с ответом |
| [6] разбор ошибок | останется пересказ теории без доказательств |
| [7] SQL/BigQuery | расчёт не воспроизводится вне вашего ноутбука |

Шаги [1]–[6] — задания 1–8 ниже, шаг [7] — задание 9 (блок BigQuery).

## 1. Настройка окружения и вариант

In [ ]:
# ВЫДАНО: установка зависимостей и импорты
import importlib
import re
import subprocess
import sys
import warnings
import zlib

def ensure(packages):
    for module, pip_name in packages.items():
        try:
            importlib.import_module(module)
        except ImportError:
            print(f"[install] {pip_name} ...")
            try:
                subprocess.run([sys.executable, "-m", "pip", "install", "-q", pip_name],
                               check=True, timeout=300)
            except Exception as exc:                  # noqa: BLE001
                print(f"[warn] не удалось установить {pip_name}: {exc}")

ensure({"rank_bm25": "rank-bm25", "sklearn": "scikit-learn",
        "pandas": "pandas", "numpy": "numpy", "matplotlib": "matplotlib"})
ensure({"kagglehub": "kagglehub"})

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import ndcg_score

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
pd.set_option("display.width", 140)
pd.set_option("display.max_colwidth", 90)
warnings.filterwarnings("ignore", category=FutureWarning)
print("Окружение готово. Python", sys.version.split()[0])

In [ ]:
# ВЫДАНО: параметры индивидуального варианта
B_GRID = [0.60, 0.70, 0.75, 0.80, 0.85]

def get_variant(v: int) -> dict:
    if not 1 <= int(v) <= 25:
        raise ValueError("Номер варианта должен быть в диапазоне 1..25")
    v = int(v)
    return {"variant": v,
            "k1": round(1.2 + 0.2 * ((v - 1) // 5), 1),
            "b": B_GRID[(v - 1) % 5],
            "k": 5 if v % 2 == 1 else 10}

# ------------------------------------------------------------------
VARIANT = 1          # <-- ЗАМЕНИТЕ на номер своего варианта (1..25)
# ------------------------------------------------------------------
PARAMS = get_variant(VARIANT)
K1, B, K = PARAMS["k1"], PARAMS["b"], PARAMS["k"]
print(f"Вариант {VARIANT}: k1 = {K1}, b = {B}, метрика NDCG@{K}")

## 2. Данные: Microsoft Research WikiQA Corpus

Единица **документа** в этой работе — одно предложение-кандидат (`Sentence`),
**запрос** — англоязычный вопрос (`Question`), **истинная релевантность** — `Label` (0/1).

Схема исходного TSV-файла:

| Поле | Смысл |
|---|---|
| `QuestionID` | идентификатор вопроса (`Q1`, `Q2`, …) |
| `Question` | текст вопроса на английском языке |
| `DocumentID`, `DocumentTitle` | статья Wikipedia, из которой взяты кандидаты |
| `SentenceID` | идентификатор предложения-кандидата |
| `Sentence` | текст предложения-кандидата |
| `Label` | 1 — предложение является ответом, 0 — не является |

Ячейка ниже реализует **три пути загрузки** (выполняются по очереди, до первого успешного):

1. локальный файл `data/wikiqa/WikiQA-test.tsv` (скачан вручную с Microsoft Download Center или с Kaggle);
2. автоматическая загрузка через `kagglehub` (зеркало `saurabhshahane/wikiqa-corpus`) или через `datasets` (`microsoft/wiki_qa`);
3. **синтетический генератор-заглушка** — на случай, когда сети нет вообще.

> ⚠️ Третий путь нужен только для того, чтобы ноутбук гарантированно запускался и код можно было отладить.
> Числа, полученные на синтетических данных, **не являются** результатом лабораторной работы:
> сдавать нужно прогон на настоящем WikiQA (переменная `DATA_SOURCE` должна быть `local`, `kagglehub` или `huggingface`).

In [ ]:
# ============================================================================
#  Загрузка WikiQA: три пути (локальный файл -> сеть -> синтетическая заглушка)
# ============================================================================
import os
import glob
import random
import warnings

import numpy as np
import pandas as pd

WIKIQA_COLUMNS = ["QuestionID", "Question", "DocumentID", "DocumentTitle",
                  "SentenceID", "Sentence", "Label"]

# Справочные размеры настоящего корпуса (Yang, Yih, Meek, EMNLP 2015)
WIKIQA_REFERENCE = {
    "train": {"questions": 2118, "sentences": 20360, "answers": 1040},
    "dev":   {"questions": 296,  "sentences": 2733,  "answers": 140},
    "test":  {"questions": 633,  "sentences": 6165,  "answers": 293},
}

DATA_SOURCE = None  # заполняется функцией load_wikiqa()


def _normalize(df):
    """Приводит датафрейм к каноническому набору колонок WIKIQA_COLUMNS."""
    df = df.copy()
    rename = {"question_id": "QuestionID", "question": "Question",
              "document_title": "DocumentTitle", "answer": "Sentence",
              "label": "Label", "sentence": "Sentence"}
    df = df.rename(columns={c: rename[c] for c in df.columns if c in rename})
    if "DocumentID" not in df.columns:
        df["DocumentID"] = df.get("DocumentTitle", pd.Series(["D0"] * len(df)))
    if "SentenceID" not in df.columns:
        df["SentenceID"] = [f"{q}-{i}" for i, q in enumerate(df["QuestionID"])]
    df["Label"] = df["Label"].astype(int)
    for col in ("Question", "Sentence", "DocumentTitle"):
        df[col] = df[col].astype(str)
    return df[WIKIQA_COLUMNS].reset_index(drop=True)


def _load_local(split, data_dir):
    """Путь 1. Локальный TSV-файл из официального архива WikiQACorpus."""
    patterns = [
        os.path.join(data_dir, f"WikiQA-{split}.tsv"),
        os.path.join(data_dir, "**", f"WikiQA-{split}.tsv"),
        os.path.join("**", f"WikiQA-{split}.tsv"),
    ]
    for pattern in patterns:
        hits = sorted(glob.glob(pattern, recursive=True))
        if hits:
            return _normalize(pd.read_csv(hits[0], sep="\t", quoting=3)), hits[0]
    return None, None


def _load_kagglehub(split):
    """Путь 2а. Kaggle-зеркало (kagglehub скачивает архив в локальный кэш)."""
    import kagglehub
    path = kagglehub.dataset_download("saurabhshahane/wikiqa-corpus")
    hits = sorted(glob.glob(os.path.join(path, "**", f"WikiQA-{split}.tsv"),
                            recursive=True))
    if not hits:
        raise FileNotFoundError(f"WikiQA-{split}.tsv не найден в {path}")
    return _normalize(pd.read_csv(hits[0], sep="\t", quoting=3)), hits[0]


def _load_huggingface(split):
    """Путь 2б. Hugging Face Hub: microsoft/wiki_qa."""
    from datasets import load_dataset
    hf_split = {"dev": "validation"}.get(split, split)
    ds = load_dataset("microsoft/wiki_qa", split=hf_split)
    return _normalize(ds.to_pandas()), "huggingface:microsoft/wiki_qa"


# ---------------------------------------------------------------------------
#  Путь 3. Синтетическая заглушка, имитирующая структуру и статистику WikiQA
# ---------------------------------------------------------------------------
_SYN_TOPICS = [
    ("Glacier cave", ["glacier", "meltwater", "crevasse", "ice", "tunnel"]),
    ("Harpsichord", ["keyboard", "plectrum", "strings", "baroque", "jack"]),
    ("Monsoon", ["rainfall", "wind", "seasonal", "tropics", "humidity"]),
    ("Lighthouse", ["beacon", "coast", "lantern", "keeper", "navigation"]),
    ("Penicillin", ["antibiotic", "mould", "bacteria", "infection", "culture"]),
    ("Sundial", ["gnomon", "shadow", "hours", "dial", "solar"]),
    ("Aqueduct", ["channel", "water", "arches", "roman", "gradient"]),
    ("Kiwi bird", ["flightless", "burrow", "nocturnal", "beak", "forest"]),
    ("Tundra", ["permafrost", "arctic", "lichen", "shrub", "soil"]),
    ("Zeppelin", ["airship", "hydrogen", "rigid", "hull", "flight"]),
    ("Sonar", ["acoustic", "echo", "submarine", "pulse", "underwater"]),
    ("Kimchi", ["fermented", "cabbage", "korean", "chilli", "jar"]),
    ("Obsidian", ["volcanic", "glass", "lava", "fracture", "blade"]),
    ("Morse code", ["telegraph", "dots", "dashes", "signal", "operator"]),
    ("Coral reef", ["polyp", "calcium", "lagoon", "tropical", "bleaching"]),
    ("Windmill", ["sails", "grain", "milling", "rotor", "tower"]),
    ("Vaccine", ["immunity", "antigen", "dose", "clinical", "virus"]),
    ("Cuneiform", ["clay", "tablet", "stylus", "sumerian", "script"]),
    ("Geyser", ["eruption", "groundwater", "steam", "vent", "basin"]),
    ("Espresso", ["pressure", "portafilter", "crema", "roast", "grind"]),
    ("Bagpipes", ["drone", "chanter", "reed", "highland", "bag"]),
    ("Permaculture", ["agriculture", "design", "guild", "mulch", "yield"]),
    ("Semaphore", ["flags", "visual", "signalling", "arms", "station"]),
    ("Plate tectonics", ["lithosphere", "subduction", "mantle", "drift", "fault"]),
    ("Origami", ["folding", "paper", "crease", "model", "japanese"]),
    ("Quinine", ["malaria", "bark", "alkaloid", "tonic", "treatment"]),
    ("Lacrosse", ["stick", "netting", "team", "goal", "field"]),
    ("Pidgin language", ["contact", "grammar", "trade", "speakers", "simplified"]),
    ("Cartography", ["projection", "scale", "map", "survey", "legend"]),
    ("Hovercraft", ["cushion", "skirt", "fan", "amphibious", "craft"]),
]

_SYN_ASPECTS = [
    ("origin", "how is the {f} of a {t} formed", ["formed", "formation", "forms"],
     ["arises", "develops", "emerges"]),
    ("definition", "what is the {f} of {t}", ["term", "refers", "defined"],
     ["denotes", "describes", "means"]),
    ("location", "where is the {f} of {t} found", ["located", "situated", "region"],
     ["lies", "occurs", "distributed"]),
    ("time", "when was the {f} of {t} first described", ["first", "described", "century"],
     ["initially", "documented", "recorded"]),
    ("person", "who invented the {t} {f}", ["invented", "inventor", "engineer"],
     ["devised", "pioneer", "credited"]),
    ("size", "how big is the {f} of a {t}", ["metres", "diameter", "large"],
     ["measures", "span", "extent"]),
    ("purpose", "what is the {f} of {t} used for", ["used", "purpose", "application"],
     ["serves", "employed", "role"]),
    ("cause", "what causes the {f} of {t}", ["caused", "cause", "results"],
     ["triggered", "produced", "driven"]),
]

_SYN_FILLER = (
    "the of and in to a is was for with on as by are from that at it this "
    "which or an be were has have been also often usually most many some "
    "such other than more into between during after before over under "
    "modern early late common known called including example although "
    "however typically generally particularly widely small large narrow "
    "wide long short heavy light natural artificial local regional "
    "surface material structure system process design method number "
    "period record study report century year time part group area"
).split()


def _syn_sentence(rng, title_tokens, topic_words, cues, length):
    """Собирает предложение заданной длины из тематических и служебных слов."""
    tokens = list(title_tokens) + list(topic_words) + list(cues)
    while len(tokens) < length:
        # частотный «хвост»: короткие служебные слова встречаются чаще
        idx = min(int(abs(rng.gauss(0, 0.35)) * len(_SYN_FILLER)), len(_SYN_FILLER) - 1)
        tokens.append(_SYN_FILLER[idx])
    rng.shuffle(tokens)
    text = " ".join(tokens)
    return text[0].upper() + text[1:] + " ."


def _load_synthetic(split, seed=20261014):
    """Путь 3. Детерминированная имитация WikiQA со схожей статистикой."""
    ref = WIKIQA_REFERENCE.get(split, WIKIQA_REFERENCE["test"])
    n_questions = ref["questions"]
    rng = random.Random(seed + len(split))
    rows = []
    for qi in range(n_questions):
        title, topic_vocab = _SYN_TOPICS[rng.randrange(len(_SYN_TOPICS))]
        aspect, tmpl, cues, syn_cues = _SYN_ASPECTS[rng.randrange(len(_SYN_ASPECTS))]
        title_tokens = title.lower().split()
        focus = rng.choice(topic_vocab)               # содержательное слово вопроса
        others = [w for w in topic_vocab if w != focus]
        question = tmpl.format(t=title.lower(), f=focus)
        # доля вопросов без единого правильного ответа ~ как в настоящем WikiQA
        has_answer = rng.random() < (243 / 633)
        n_cand = max(2, int(rng.gauss(9.7, 3.5)))
        n_pos = rng.randint(1, 2) if has_answer else 0
        pos_slots = set(rng.sample(range(n_cand), k=min(n_pos, n_cand)))
        # четверть правильных ответов выражена синонимами, а не словами вопроса
        paraphrase = rng.random() < 0.25
        for si in range(n_cand):
            topic_words = rng.sample(others, k=min(len(others), rng.randint(1, 3)))
            if si in pos_slots:
                label = 1
                if paraphrase:                        # лексический разрыв
                    sent_cues = rng.sample(syn_cues, k=min(2, len(syn_cues)))
                else:
                    sent_cues = rng.sample(cues, k=min(2, len(cues))) + [focus]
            else:
                label = 0
                sent_cues = []
                if rng.random() < 0.30:               # «трудный» отрицательный
                    sent_cues.append(rng.choice(cues))
                if rng.random() < 0.25:
                    sent_cues.append(focus)
                if rng.random() < 0.12:               # «переспам»: слово вопроса 2-4 раза
                    sent_cues += [focus] * rng.randint(2, 4)
            length = max(6, int(rng.gauss(25, 11)))   # средняя длина ~25 токенов
            rows.append({
                "QuestionID": f"S{qi + 1}",
                "Question": question,
                "DocumentID": f"D{qi + 1}",
                "DocumentTitle": title,
                "SentenceID": f"S{qi + 1}-{si}",
                "Sentence": _syn_sentence(rng, title_tokens, topic_words,
                                          sent_cues, length),
                "Label": label,
            })
    return _normalize(pd.DataFrame(rows)), "synthetic"


def load_wikiqa(split="test", data_dir="data/wikiqa", allow_synthetic=True,
                verbose=True):
    """Возвращает (DataFrame, источник). Порядок: локально -> сеть -> заглушка."""
    global DATA_SOURCE
    attempts = [
        ("local", lambda: _load_local(split, data_dir)),
        ("kagglehub", lambda: _load_kagglehub(split)),
        ("huggingface", lambda: _load_huggingface(split)),
    ]
    for name, fn in attempts:
        try:
            df, origin = fn()
            if df is not None and len(df) > 0:
                DATA_SOURCE = name
                if verbose:
                    print(f"[OK] Источник данных: {name} ({origin})")
                return df, name
        except Exception as exc:                      # noqa: BLE001
            if verbose:
                print(f"[--] {name}: {type(exc).__name__}: {exc}")
    if not allow_synthetic:
        raise RuntimeError(
            "Настоящий WikiQA не найден. Скачайте WikiQACorpus.zip и положите "
            "WikiQA-test.tsv в data/wikiqa/, либо установите kagglehub/datasets."
        )
    df, _ = _load_synthetic(split)
    DATA_SOURCE = "synthetic"
    warnings.warn("Используется СИНТЕТИЧЕСКАЯ заглушка: результаты не засчитываются "
                  "как выполнение лабораторной работы.", stacklevel=2)
    if verbose:
        print("[!!] Источник данных: synthetic (заглушка, только для отладки кода)")
    return df, "synthetic"


def describe_corpus(df, split="test"):
    """Печатает статистику загруженного корпуса и сверяет её со справочной."""
    n_q = df["QuestionID"].nunique()
    n_s = len(df)
    n_a = int(df["Label"].sum())
    per_q = df.groupby("QuestionID")["Label"].agg(["size", "sum"])
    n_q_with_answer = int((per_q["sum"] > 0).sum())
    print(f"Вопросов:                      {n_q}")
    print(f"Предложений-кандидатов:        {n_s}")
    print(f"Предложений с Label=1:         {n_a}")
    print(f"Вопросов, где есть ответ:      {n_q_with_answer} "
          f"({n_q_with_answer / max(n_q, 1):.1%})")
    print(f"Кандидатов на вопрос (сред.):  {per_q['size'].mean():.2f} "
          f"[{per_q['size'].min()}..{per_q['size'].max()}]")
    print(f"Длина предложения (токенов):   "
          f"{df['Sentence'].str.split().str.len().mean():.2f}")
    ref = WIKIQA_REFERENCE.get(split)
    if ref:
        ok = (n_q == ref["questions"] and n_s == ref["sentences"]
              and n_a == ref["answers"])
        print(f"\nСправочные значения WikiQA-{split}: "
              f"{ref['questions']} вопросов / {ref['sentences']} предложений / "
              f"{ref['answers']} ответов")
        print("Совпадение со справочными значениями:", "ДА" if ok else "НЕТ")
    return per_q

In [ ]:
# ВЫДАНО: загрузка корпуса и базовая статистика
SPLIT = "test"
df, source = load_wikiqa(split=SPLIT, data_dir="data/wikiqa", allow_synthetic=True)
df = df.reset_index(drop=True)
describe_corpus(df, split=SPLIT)
df.head(5)

## 2. Задание 1. Исследование candidate sets

Ответьте на вопросы (числами, полученными из данных, а не из README):

1. Сколько всего вопросов и сколько предложений-кандидатов в загруженном split?
2. У какой доли вопросов есть хотя бы одно предложение с `Label = 1`?
3. Каково среднее число кандидатов на вопрос и средняя длина предложения?
4. Выведите все кандидаты одного вопроса, у которого есть правильный ответ,
   и пометьте правильные предложения.

Почему это важно: все кандидаты одного вопроса взяты из одной и той же статьи
Wikipedia, поэтому слова из заголовка статьи почти не различают кандидатов.

In [ ]:
# TODO 1: разведочный анализ candidate sets
# Подсказка: df.groupby("QuestionID").agg(...)

stats = None        # TODO: таблица со статистикой по каждому вопросу
# TODO: напечатайте ответы на вопросы 1–3
# TODO: выведите кандидатов одного вопроса с пометкой правильных ответов

## 3. Задание 2. Единая токенизация

`rank_bm25` **не выполняет предобработку самостоятельно**: запрос нужно обрабатывать
точно так же, как документы. Поэтому напишем один токенизатор и будем передавать его
и в `TfidfVectorizer` (через параметр `analyzer`), и в BM25.

Требования к `tokenize`:

* приводит текст к нижнему регистру;
* оставляет только последовательности букв и цифр (пунктуация отбрасывается);
* при `remove_stopwords=True` убирает английские стоп-слова из `ENGLISH_STOPWORDS`
  (вопросительные слова *how, what, when, who* в этот список намеренно **не** включены —
  посмотрите в разборе запросов, помогают они или мешают);
* возвращает **список** токенов.

In [ ]:
# ВЫДАНО: список стоп-слов и заготовка регулярного выражения
TOKEN_RE = re.compile(r"[a-z0-9]+")

ENGLISH_STOPWORDS = frozenset("""
a an the and or but if then than that this these those there here of in on at to
for with without within from by as is are was were be been being am do does did
doing have has had having will would shall should can could may might must
it its it's he she they them his her their our your my me we you i not no nor so
such about into over under again further once during before after above below
""".split())


# TODO 2: реализуйте токенизацию
def tokenize(text, remove_stopwords: bool = True, stem: bool = False):
    """Возвращает список токенов для текста (одинаково для запросов и документов)."""
    # TODO: приведите к нижнему регистру и примените TOKEN_RE
    # TODO: при remove_stopwords=True отфильтруйте стоп-слова
    # (параметр stem пока можно игнорировать — он понадобится в повышенной части)
    raise NotImplementedError("Задание 2: реализуйте tokenize()")


def make_tokenizer(remove_stopwords=True, stem=False):
    """ВЫДАНО: фабрика токенизаторов для TfidfVectorizer и BM25."""
    def _tok(text):
        return tokenize(text, remove_stopwords=remove_stopwords, stem=stem)
    return _tok


TOKENIZER = make_tokenizer(remove_stopwords=True, stem=False)

In [ ]:
# ВЫДАНО: автотест задания 2
def test_tokenize():
    assert tokenize("The Glacier Caves were formed!") == ["glacier", "caves", "formed"], \
        "проверьте нижний регистр, пунктуацию и удаление стоп-слов"
    assert tokenize("The 1889 Paris Exposition!") == ["1889", "paris", "exposition"], \
        "цифры должны сохраняться"
    assert tokenize("the of and in to") == [], "стоп-слова должны удаляться"
    assert tokenize("The of and", remove_stopwords=False) == ["the", "of", "and"], \
        "при remove_stopwords=False стоп-слова остаются"
    assert isinstance(tokenize("test"), list), "функция должна возвращать список"
    assert TOKENIZER("A lighthouse.") == tokenize("A lighthouse."), \
        "TOKENIZER должен совпадать с tokenize"
    print("test_tokenize: OK")

test_tokenize()

## 4. Задание 3. TF-IDF

`TfidfVectorizer` строит матрицу «документ × терм». При `norm="l2"` строки нормированы,
поэтому скалярное произведение вектора вопроса и вектора предложения равно косинусной
близости:

$$\cos(q,d)=\frac{\langle q,d\rangle}{\lVert q\rVert\lVert d\rVert}=\langle\hat q,\hat d\rangle$$

**Важно:** индекс строится по **всем** предложениям split (чтобы IDF был
общекорпусным), а ранжируются только кандидаты конкретного вопроса.

In [ ]:
# TODO 3: постройте индекс TF-IDF и функцию оценки
SENTENCES = df["Sentence"].tolist()

tfidf_vectorizer = None     # TODO: TfidfVectorizer(analyzer=TOKENIZER, norm="l2")
TFIDF_MATRIX = None         # TODO: fit_transform по SENTENCES


def tfidf_scorer(question, candidate_idx):
    """Косинусная близость вопроса и кандидатов.

    question      : str — текст вопроса
    candidate_idx : массив номеров строк df (глобальные индексы кандидатов)
    return        : np.ndarray оценок той же длины, что candidate_idx
    """
    # TODO: преобразуйте вопрос вектором tfidf_vectorizer.transform([question])
    # TODO: перемножьте строки TFIDF_MATRIX[candidate_idx] с вектором вопроса
    raise NotImplementedError("Задание 3: реализуйте tfidf_scorer()")

In [ ]:
# ВЫДАНО: автотест задания 3
def test_tfidf():
    assert TFIDF_MATRIX is not None and TFIDF_MATRIX.shape[0] == len(df), \
        "индекс должен быть построен по ВСЕМ предложениям split"
    qid = df.loc[df["Label"] == 1, "QuestionID"].iloc[0]
    idx = df.index[df["QuestionID"] == qid].to_numpy()
    scores = np.asarray(tfidf_scorer(df.loc[idx[0], "Question"], idx), dtype=float)
    assert scores.shape == (len(idx),), "длина ответа должна совпадать с числом кандидатов"
    assert np.all(scores >= -1e-9) and np.all(scores <= 1 + 1e-9), \
        "косинусная близость неотрицательных векторов лежит в [0, 1]"
    self_query = df.loc[idx[0], "Sentence"]          # запрос = сам первый кандидат
    self_scores = np.asarray(tfidf_scorer(self_query, idx), dtype=float)
    assert int(np.argmax(self_scores)) == 0, \
        "если запрос дословно совпал с кандидатом, он должен получить максимальную оценку"
    print("test_tfidf: OK")

test_tfidf()

### Служебный код: векторизованный BM25

Ячейка ниже — **выдана готовой**, изменять её не нужно. Она решает две задачи:

* даёт быструю (матричную) реализацию BM25 Okapi, с помощью которой считаются
  «сетки» по параметрам $k_1, b$ — библиотечный `BM25Okapi` пришлось бы вызывать
  тысячи раз, и это было бы медленно;
* служит **аварийным запасным вариантом**, если `rank_bm25` не установился
  (например, в аудитории нет интернета).

Формула здесь та же, что и в `rank_bm25`, включая обработку отрицательных IDF:

$$\mathrm{IDF}(q)=\ln\frac{N-n(q)+0.5}{n(q)+0.5},\qquad
\mathrm{BM25}(D,Q)=\sum_{q\in Q}\mathrm{IDF}(q)\,
\frac{f(q,D)\,(k_1+1)}{f(q,D)+k_1\left(1-b+b\frac{|D|}{\mathrm{avgdl}}\right)}$$

Совпадение с библиотекой проверяется автотестом ниже (расхождение должно быть < 1e-9).

In [ ]:
# ============================================================================
#  Служебный код (выдан готовым): векторизованный BM25 Okapi
# ============================================================================
import numpy as np
from scipy import sparse
from sklearn.feature_extraction.text import CountVectorizer

try:
    from rank_bm25 import BM25Okapi
    HAS_RANK_BM25 = True
except ImportError:                                   # pragma: no cover
    BM25Okapi = None
    HAS_RANK_BM25 = False
    print("[!] Библиотека rank_bm25 не установлена: используется запасная "
          "реализация BM25Fast. Для сдачи работы установите rank-bm25.")


class BM25Fast:
    """BM25 Okapi на разреженной матрице частот. Совместим с rank_bm25.BM25Okapi.

    Parameters
    ----------
    corpus : list[str]           тексты документов (предложений-кандидатов)
    tokenizer : callable         та же токенизация, что и для запросов
    k1, b, epsilon : float       параметры BM25 (epsilon — как в rank_bm25)
    """

    def __init__(self, corpus, tokenizer, k1=1.5, b=0.75, epsilon=0.25):
        self.k1, self.b, self.epsilon = float(k1), float(b), float(epsilon)
        self.tokenizer = tokenizer
        self.vectorizer = CountVectorizer(analyzer=tokenizer)
        self.tf = self.vectorizer.fit_transform(corpus).tocsr().astype(np.float64)
        self.vocab = self.vectorizer.vocabulary_
        self.doc_len = np.asarray(self.tf.sum(axis=1)).ravel()
        self.avgdl = float(self.doc_len.mean())
        n_docs = self.tf.shape[0]
        doc_freq = np.asarray((self.tf > 0).sum(axis=0)).ravel().astype(np.float64)
        idf = np.log(n_docs - doc_freq + 0.5) - np.log(doc_freq + 0.5)
        average_idf = float(idf.mean())
        idf[idf < 0] = self.epsilon * average_idf      # как в rank_bm25
        self.idf = idf
        self._denom_len = self.k1 * (1.0 - self.b + self.b * self.doc_len / self.avgdl)

    def _query_columns(self, query_tokens):
        return [self.vocab[t] for t in query_tokens if t in self.vocab]

    def score_candidates(self, query, candidate_idx):
        """Оценки BM25 только для указанных строк корпуса (быстрый путь)."""
        cols = self._query_columns(self.tokenizer(query))
        candidate_idx = np.asarray(candidate_idx, dtype=int)
        if not cols:
            return np.zeros(len(candidate_idx))
        sub = self.tf[candidate_idx][:, cols].toarray()          # (n_cand, n_terms)
        denom = sub + self._denom_len[candidate_idx][:, None]
        contrib = np.asarray(self.idf)[cols][None, :] * sub * (self.k1 + 1.0) / denom
        return contrib.sum(axis=1)

    def get_scores(self, query_tokens):
        """Оценки для всего корпуса — сигнатура как у rank_bm25.BM25Okapi."""
        cols = self._query_columns(query_tokens)
        if not cols:
            return np.zeros(self.tf.shape[0])
        sub = self.tf[:, cols].toarray()
        denom = sub + self._denom_len[:, None]
        return (np.asarray(self.idf)[cols][None, :] * sub * (self.k1 + 1.0) / denom).sum(axis=1)

## 5. Задание 4. BM25

$$\mathrm{BM25}(D,Q)=\sum_{q\in Q}\mathrm{IDF}(q)\cdot
\frac{f(q,D)\,(k_1+1)}{f(q,D)+k_1\left(1-b+b\dfrac{\lvert D\rvert}{\mathrm{avgdl}}\right)}$$

Постройте **два** индекса:

* `bm25_default` с параметрами по умолчанию $k_1=1.5$, $b=0.75$;
* `bm25_variant` с параметрами вашего варианта $k_1$, $b$.

Второй индекс нужен, чтобы отделить эффект вашего варианта от эффекта самого
перехода TF-IDF → BM25.

Подсказки:

* `BM25Okapi` принимает корпус как **список списков токенов**, а не список строк;
* `get_scores(query_tokens)` возвращает оценки для **всего** корпуса — нужные
  кандидаты отбираются индексацией;
* запрос токенизируется тем же `TOKENIZER`.

In [ ]:
# TODO 4: постройте BM25-индексы и функции оценки
CORPUS_TOKENS = None        # TODO: список списков токенов по SENTENCES

bm25_default = None         # TODO: BM25Okapi(CORPUS_TOKENS, k1=1.5, b=0.75)
bm25_variant = None         # TODO: BM25Okapi(CORPUS_TOKENS, k1=K1, b=B)


def bm25_default_scorer(question, candidate_idx):
    """Оценки BM25 (k1=1.5, b=0.75) для кандидатов."""
    raise NotImplementedError("Задание 4: реализуйте bm25_default_scorer()")


def bm25_variant_scorer(question, candidate_idx):
    """Оценки BM25 с параметрами вашего варианта."""
    raise NotImplementedError("Задание 4: реализуйте bm25_variant_scorer()")

In [ ]:
# ВЫДАНО: автотест задания 4 (сверка с эталонной векторизованной реализацией)
def test_bm25():
    reference = BM25Fast(SENTENCES, TOKENIZER, k1=K1, b=B)
    reference_default = BM25Fast(SENTENCES, TOKENIZER, k1=1.5, b=0.75)
    qids = df.loc[df["Label"] == 1, "QuestionID"].drop_duplicates().head(15)
    max_diff_v = max_diff_d = 0.0
    for qid in qids:
        idx = df.index[df["QuestionID"] == qid].to_numpy()
        q = df.loc[idx[0], "Question"]
        got_v = np.asarray(bm25_variant_scorer(q, idx), dtype=float)
        got_d = np.asarray(bm25_default_scorer(q, idx), dtype=float)
        assert got_v.shape == (len(idx),), "длина ответа не совпала с числом кандидатов"
        max_diff_v = max(max_diff_v, float(np.abs(got_v - reference.score_candidates(q, idx)).max()))
        max_diff_d = max(max_diff_d, float(np.abs(got_d - reference_default.score_candidates(q, idx)).max()))
    assert max_diff_v < 1e-9, (
        f"BM25 варианта расходится с эталоном (Δ={max_diff_v:.2e}): проверьте k1={K1}, "
        f"b={B} и токенизацию запроса")
    assert max_diff_d < 1e-9, (
        f"BM25 по умолчанию расходится с эталоном (Δ={max_diff_d:.2e})")
    print(f"test_bm25: OK (максимальное расхождение с эталоном {max(max_diff_v, max_diff_d):.1e})")

test_bm25()

## 6. Задание 5. DCG и NDCG

$$DCG@k=\sum_{i=1}^{k}\frac{2^{rel_i}-1}{\log_2(i+1)},\qquad
NDCG@k=\frac{DCG@k}{IDCG@k}$$

$IDCG@k$ — значение $DCG@k$ при идеальном порядке (все релевантные наверху).

Договорённость: если у вопроса **нет** ни одного релевантного документа, $IDCG@k=0$,
NDCG не определён — функция возвращает `float("nan")`, и такие вопросы исключаются
из усреднения. Заменять их нулём нельзя: тогда метрика начнёт измерять долю вопросов
без ответа, а не качество ранжирования.

In [ ]:
# TODO 5: реализуйте метрики
def dcg_at_k(relevances, k):
    """DCG@k для последовательности релевантностей В ПОРЯДКЕ ВЫДАЧИ."""
    # TODO: возьмите первые k элементов, посчитайте (2**rel - 1) / log2(i + 1)
    raise NotImplementedError("Задание 5: реализуйте dcg_at_k()")


def ndcg_at_k(relevances, k):
    """NDCG@k; float('nan'), если релевантных документов нет."""
    # TODO: постройте идеальный порядок (сортировка по убыванию) и поделите DCG на IDCG
    raise NotImplementedError("Задание 5: реализуйте ndcg_at_k()")

In [ ]:
# ВЫДАНО: автотест задания 5 (ручные примеры + сверка с scikit-learn)
def test_metrics():
    assert abs(dcg_at_k([1, 0, 0], 3) - 1.0) < 1e-12, "релевантный на 1-й позиции даёт 1.0"
    assert abs(dcg_at_k([0, 1, 0], 3) - 1 / np.log2(3)) < 1e-12, "проверьте дисконт log2(i+1)"
    assert abs(dcg_at_k([1, 1], 1) - 1.0) < 1e-12, "должны учитываться только первые k позиций"
    assert abs(ndcg_at_k([1, 0, 0, 0], 4) - 1.0) < 1e-12, "идеальный порядок даёт NDCG = 1"
    assert abs(ndcg_at_k([0, 0, 1], 2) - 0.0) < 1e-12, "релевантный ниже отсечки k"
    assert np.isnan(ndcg_at_k([0, 0, 0], 3)), "при отсутствии релевантных верните nan"

    rng = np.random.default_rng(0)
    worst = 0.0
    for _ in range(200):
        n = int(rng.integers(3, 12))
        rel = rng.integers(0, 2, size=n)
        if rel.sum() == 0:
            continue
        scores = rng.normal(size=n)
        order = np.argsort(-scores, kind="stable")
        for kk in (5, 10):
            worst = max(worst, abs(ndcg_at_k(rel[order], kk)
                                   - ndcg_score(rel[None, :], scores[None, :], k=kk)))
    assert worst < 1e-9, f"расхождение с sklearn.metrics.ndcg_score: {worst:.2e}"
    print(f"test_metrics: OK (максимальное расхождение с sklearn {worst:.1e})")

test_metrics()

## 7. Схема оценивания (выдана готовой)

Функция ранжирования получает только текст вопроса и номера строк кандидатов —
меток она не видит. Это структурная защита от утечки разметки; автотест ниже
дополнительно проверяет, что оценки не меняются при перемешивании `Label`.

In [ ]:
# ВЫДАНО: группировка запросов, функция оценивания, детектор утечки
def build_groups(frame):
    groups = []
    for qid, g in frame.groupby("QuestionID", sort=False):
        groups.append({"qid": qid, "question": g["Question"].iloc[0],
                       "title": g["DocumentTitle"].iloc[0],
                       "idx": g.index.to_numpy(), "labels": g["Label"].to_numpy()})
    return groups


def evaluate(scorer, groups, k, per_query=False):
    """Средние DCG@k и NDCG@k по запросам, у которых есть хотя бы один Label=1."""
    rows = []
    for g in groups:
        if g["labels"].sum() == 0:
            continue
        scores = np.asarray(scorer(g["question"], g["idx"]), dtype=float)
        order = np.argsort(-scores, kind="stable")
        rels = g["labels"][order]
        rows.append({"QuestionID": g["qid"], "question": g["question"],
                     "dcg": dcg_at_k(rels, k), "ndcg": ndcg_at_k(rels, k)})
    result = pd.DataFrame(rows)
    if per_query:
        return result
    return {"queries": len(result), "mean_DCG": result["dcg"].mean(),
            "mean_NDCG": result["ndcg"].mean()}


def check_no_label_leakage(scorer, groups, n=25, seed=RANDOM_SEED):
    rng_local = np.random.default_rng(seed)
    sample = [groups[i] for i in rng_local.choice(len(groups), size=min(n, len(groups)),
                                                  replace=False)]
    before = [np.asarray(scorer(g["question"], g["idx"]), dtype=float) for g in sample]
    shuffled = [dict(g, labels=rng_local.permutation(g["labels"])) for g in sample]
    after = [np.asarray(scorer(g["question"], g["idx"]), dtype=float) for g in shuffled]
    return all(np.allclose(a, b) for a, b in zip(before, after))


GROUPS = build_groups(df)
ELIGIBLE = [g for g in GROUPS if g["labels"].sum() > 0]
print(f"Всего запросов: {len(GROUPS)}; пригодных для NDCG: {len(ELIGIBLE)} "
      f"({len(ELIGIBLE) / len(GROUPS):.1%})")

assert check_no_label_leakage(tfidf_scorer, ELIGIBLE), "TF-IDF использует Label — это утечка"
assert check_no_label_leakage(bm25_variant_scorer, ELIGIBLE), "BM25 использует Label — это утечка"
print("Проверка на утечку разметки: OK")

In [ ]:
# ВЫДАНО: наивные ориентиры, с которыми сравниваются TF-IDF и BM25
_counts = BM25Fast(SENTENCES, TOKENIZER)      # индекс частот строится один раз
COUNT_MATRIX, COUNT_VOCAB = _counts.tf, _counts.vocab


def random_scorer(question, candidate_idx):
    """Случайный порядок (детерминированный по тексту вопроса)."""
    seed = zlib.crc32(question.encode("utf-8")) % (2 ** 32)
    return np.random.default_rng(seed).random(len(candidate_idx))


def overlap_scorer(question, candidate_idx):
    """Сумма частот терминов запроса в кандидате: без IDF и без нормировки длины."""
    cols = [COUNT_VOCAB[t] for t in TOKENIZER(question) if t in COUNT_VOCAB]
    candidate_idx = np.asarray(candidate_idx)
    if not cols:
        return np.zeros(len(candidate_idx))
    return np.asarray(COUNT_MATRIX[candidate_idx][:, cols].sum(axis=1)).ravel().astype(float)

## 8. Задание 6. Сводная таблица

Постройте таблицу со строками `random`, `overlap`, `TF-IDF`, `BM25 default (1.5, 0.75)`,
`BM25 variant` и колонками `queries`, `mean_DCG@k`, `mean_NDCG@k` для $k=5$ и $k=10$.

Строка с вашим вариантом обязательна; строка `BM25 default` нужна, чтобы отделить
эффект варианта от эффекта перехода TF-IDF → BM25.

In [ ]:
# TODO 6: сводная таблица сравнения методов
METHODS = [
    ("random", random_scorer),
    ("overlap", overlap_scorer),
    ("TF-IDF (cosine)", tfidf_scorer),
    ("BM25 default (k1=1.5, b=0.75)", bm25_default_scorer),
    (f"BM25 variant {VARIANT} (k1={K1}, b={B})", bm25_variant_scorer),
]

comparison = None    # TODO: pd.DataFrame со строками METHODS и k из (5, 10)
# TODO: постройте таблицу с помощью evaluate(...) и выведите её
# TODO: постройте столбчатую диаграмму mean_NDCG@k по методам

## 9. Задание 7. Разбор конкретных запросов

Найдите **не менее трёх** вопросов с наибольшим значением
$\left|NDCG_{BM25}-NDCG_{TF\text{-}IDF}\right|$ и для каждого выведите таблицу
кандидатов: `Label`, длина предложения, число совпавших с вопросом терминов,
оценки и позиции обоих методов.

Затем письменно объясните каждый случай. Возможные причины:

* насыщение частоты термина ($k_1$) — повторы одного слова перестают помогать;
* нормировка длины ($b$) — длинные предложения штрафуются иначе, чем при $L_2$;
* редкий термин (имя собственное, год, число) резко поднимает документ;
* **лексический разрыв**: правильный ответ перефразирует вопрос (синонимия,
  морфологическая вариативность, номинализация, смена залога), и оба лексических
  метода ставят его низко.

Последний пункт особенно важен для профиля «Информатика и английский язык»:
прокомментируйте примеры именно с языковой точки зрения.

In [ ]:
# TODO 7: per-query анализ расхождений
pq_tfidf = None      # TODO: evaluate(tfidf_scorer, ELIGIBLE, K, per_query=True)
pq_bm25 = None       # TODO: то же для bm25_variant_scorer
# TODO: объедините таблицы, посчитайте delta = ndcg_bm25 - ndcg_tfidf
# TODO: выведите топ-3 запроса по |delta|


def show_ranking(qid, top=8):
    """TODO: вывести подробную таблицу выдачи по одному вопросу."""
    raise NotImplementedError("Задание 7: реализуйте show_ranking()")

## 10. Задание 8. Выводы

Заполните (5–10 предложений, своими словами, со ссылкой на **свои** числа):

1. Какой метод оказался лучше при вашем $k$ и на сколько?
2. Насколько отличается ваш вариант $(k_1, b)$ от BM25 по умолчанию? Почему разница
   именно такая — вспомните, что кандидаты WikiQA являются короткими предложениями?
3. Что дают наивные ориентиры `random` и `overlap`: сколько «стоит» простое
   совпадение слов и сколько добавляет IDF?
4. Приведите один разобранный запрос и объясните расхождение методов.
5. В каких случаях лексическое ранжирование принципиально не справляется и что
   с этим можно было бы сделать?

> _Ваши выводы:_
>
> …

---

## Задание 9 (блок больших данных). Масштабирование: расчёт NDCG в Google BigQuery

> Блок выполняется **по указанию преподавателя**: либо как обязательная часть, если
> в вашем потоке изучается работа с большими данными, либо как повышенная часть
> (пункт Д, до +1 балла). Порядок действий тот же, что в практической работе № 1:
> Colab → CSV → BigQuery Sandbox → SQL с оконными функциями → сверка результатов.

### Зачем переносить расчёт в BigQuery

В лабораторной работе 6 165 строк — всё помещается в память Colab. В реальной
поисковой системе счёт идёт на миллионы пар «запрос — документ», и метрики качества
считают **рядом с данными**, не выгружая их в ноутбук.

| Что даёт перенос в BigQuery | Практический смысл |
|---|---|
| Оценки лежат в таблице, а не в памяти процесса | результат доступен коллегам без перезапуска ноутбука |
| NDCG считается оконными функциями | тот же SQL работает и на миллиардах строк |
| Метрика описана на SQL | её может проверить аналитик, не работающий с Python |
| Версии выдачи хранятся рядом | релизы поиска сравниваются одним запросом |

### Что выгружаем

Одна строка выгрузки — **один кандидат одного вопроса**: идентификаторы, порядок
предложения на странице, метка релевантности и оценки всех методов.

Тексты вопросов и предложений в выгрузку **не включаются**: для расчёта метрики они
не нужны, а объём таблицы падает на порядок. Это тот же принцип минимизации данных,
что и в практической работе № 1.

In [ ]:
# TODO 9.1: соберите таблицу оценок для выгрузки в BigQuery
import os

# Требуемые колонки (порядок значения не имеет):
#   question_id, sentence_id, cand_order, label, sent_len,
#   score_overlap, score_tfidf, score_bm25_default, score_bm25_variant
#
# Подсказки:
#   * идите по GROUPS: у каждой группы есть "qid", "question", "idx", "labels";
#   * cand_order — порядковый номер кандидата ВНУТРИ вопроса (0, 1, 2, ...);
#     он понадобится в SQL, чтобы одинаково разрешать ничьи;
#   * sent_len — длина предложения в токенах, len(TOKENIZER(...));
#   * оценки берутся у уже написанных функций *_scorer;
#   * округлите оценки до 6 знаков: round(float(value), 6).

scores_df = None      # TODO: pd.DataFrame со строкой на каждого кандидата

# TODO: сохраните таблицу в CSV без индекса
# scores_df.to_csv("wikiqa_scores.csv", index=False)

In [ ]:
# ВЫДАНО: автотест задания 9.1
def test_export():
    required = {"question_id", "sentence_id", "cand_order", "label", "sent_len",
                "score_overlap", "score_tfidf", "score_bm25_default", "score_bm25_variant"}
    assert scores_df is not None, "таблица scores_df не построена"
    missing = required - set(scores_df.columns)
    assert not missing, f"нет колонок: {sorted(missing)}"
    forbidden = {"Sentence", "Question", "sentence", "question"} & set(scores_df.columns)
    assert not forbidden, f"тексты выгружать не нужно (минимизация данных): {sorted(forbidden)}"
    assert len(scores_df) == len(df), \
        f"строк должно быть столько же, сколько кандидатов: {len(df)}, а не {len(scores_df)}"
    assert scores_df["question_id"].nunique() == df["QuestionID"].nunique(), \
        "число вопросов не совпало"
    assert int(scores_df["label"].sum()) == int(df["Label"].sum()), "сумма label не совпала"
    order = scores_df.groupby("question_id")["cand_order"].agg(["min", "max", "nunique"])
    assert (order["min"] == 0).all(), "cand_order должен начинаться с 0 внутри каждого вопроса"
    assert (order["max"] + 1 == order["nunique"]).all(), \
        "cand_order должен быть непрерывной нумерацией 0..n-1 без повторов"
    for col in ("score_overlap", "score_tfidf", "score_bm25_default", "score_bm25_variant"):
        assert np.isfinite(scores_df[col].to_numpy(dtype=float)).all(), \
            f"в колонке {col} есть NaN или inf"
    assert os.path.exists("wikiqa_scores.csv"), "файл wikiqa_scores.csv не сохранён"
    size_mb = os.path.getsize("wikiqa_scores.csv") / 1024 ** 2
    assert size_mb < 90, "файл слишком велик для веб-загрузки в BigQuery Sandbox"
    print(f"test_export: OK (строк {len(scores_df):,}, файл {size_mb:.2f} МБ)")

test_export()

### 9.2. Создание датасета и загрузка таблицы (BigQuery Sandbox)

1. Откройте `https://console.cloud.google.com/bigquery` и убедитесь, что вверху
   отображается **Sandbox**. Если проекта ещё нет — создайте его и используйте
   далее его **Project ID**.
2. В панели **Explorer** раскройте свой проект → **Create dataset**:
   * **Dataset ID:** `text_ranking`;
   * **Location type:** `Multi-region`, **Location:** `EU`.
3. Откройте `text_ranking` → **Create table**:
   * **Create table from:** `Upload`, файл `wikiqa_scores.csv`, формат `CSV`;
   * **Destination:** dataset `text_ranking`, table `wikiqa_scores`, тип `Native table`;
   * **Schema:** `Auto detect`; в **Advanced options** → **Header rows to skip = 1**.
4. Проверьте вкладку **Schema**: `cand_order`, `label`, `sent_len` должны быть
   `INTEGER`, все `score_*` — `FLOAT`. Если они определились как `STRING`, таблицу
   нужно пересоздать: иначе `ORDER BY` отсортирует числа как текст и результат
   будет неверным.
5. Нажмите **Query → In new tab**.

> В запросах ниже подставьте **свой Project ID** вместо `your-project-id`.
>
> **Ограничения Sandbox:** без `INSERT` / `UPDATE` / `DELETE` / `MERGE` и без
> streaming; таблицы имеют ограниченный срок жизни. Вся работа строится на `SELECT`.

### Проверка 0 (выполните первой). Тот ли файл загружен

```sql
SELECT
  COUNT(*)                    AS rows_count,
  COUNT(DISTINCT question_id) AS questions,
  SUM(label)                  AS positives,
  COUNT(DISTINCT IF(label = 1, question_id, NULL)) AS questions_with_answer
FROM `your-project-id.text_ranking.wikiqa_scores`;
```

Для `WikiQA-test` ожидается: `rows_count = 6165`, `questions = 633`,
`positives = 293`, `questions_with_answer = 243`. Если числа другие — дальнейшие
результаты с Python совпадать не будут, и продолжать бессмысленно.

### Пример (разобран, повторять не нужно). Ранжирование = `ROW_NUMBER()`

Сортировка кандидатов внутри одного вопроса — это оконная функция. Ничьи
разрешаются по `cand_order`, то есть так же, как устойчивая сортировка в Python.

```sql
SELECT
  question_id,
  sentence_id,
  label,
  score_bm25_variant,
  ROW_NUMBER() OVER (
    PARTITION BY question_id
    ORDER BY score_bm25_variant DESC, cand_order
  ) AS position
FROM `your-project-id.text_ranking.wikiqa_scores`
WHERE question_id = 'Q1857'
ORDER BY position;
```

### TODO 9.3. Запрос A: DCG@k и NDCG@k средствами SQL

Напишите запрос, который возвращает `queries`, `mean_dcg`, `mean_ndcg` для
**вашего** варианта BM25 при **вашем** $k$. План решения:

1. CTE `ranked`: две оконные функции — `pos` (порядок по `score_bm25_variant DESC,
   cand_order`) и `ideal_pos` (порядок по `label DESC, cand_order`);
2. CTE `per_query`: для каждого `question_id` посчитать
   `dcg  = SUM(IF(pos       <= k, (POW(2, label) - 1) / LOG(pos + 1, 2),       0))`
   и `idcg = SUM(IF(ideal_pos <= k, (POW(2, label) - 1) / LOG(ideal_pos + 1, 2), 0))`;
3. итог: `COUNT(*)`, `AVG(dcg)`, `AVG(dcg / idcg)` с условием `WHERE idcg > 0`.

Условие `WHERE idcg > 0` — это перенос в SQL того же соглашения, что и в Python:
вопросы без правильного ответа **исключаются**, а не засчитываются нулём. Если его
забыть, метрика окажется заниженной примерно до 38 % от верного значения.

Результат должен совпасть со строкой `BM25 variant` вашей таблицы из задания 6.

```sql
-- TODO: ваш запрос A
```

### TODO 9.4. Запрос B: сравнение всех методов одним запросом

Используйте `UNPIVOT`, чтобы превратить четыре колонки оценок в «длинный» формат,
и посчитайте метрику сразу для всех методов и для $k = 5$ и $k = 10$:

```sql
WITH long AS (
  SELECT question_id, cand_order, label, method, score
  FROM `your-project-id.text_ranking.wikiqa_scores`
  UNPIVOT (
    score FOR method IN (
      score_overlap, score_tfidf, score_bm25_default, score_bm25_variant
    )
  )
)
-- TODO: добавьте CTE ranked и per_query (как в запросе A, но с PARTITION BY method, question_id),
--       переберите k через CROSS JOIN UNNEST([5, 10]) AS k_value
--       и верните method, k, queries, mean_dcg, mean_ndcg
```

Сохраните результат этого запроса: **Save results → CSV**, имя файла
`bq_ndcg_result.csv`, затем загрузите его обратно в Colab.

### TODO 9.5 (по желанию). Запрос C: запросы с максимальным расхождением

Повторите средствами SQL анализ из задания 7: посчитайте `ndcg_tfidf` и
`ndcg_bm25` для каждого вопроса, выведите топ-10 по `ABS(delta)`. Сравните список
с тем, что вы получили в Python.

In [ ]:
# ВЫДАНО: сверка Python и BigQuery (задание 9.4)
BQ_RESULT = "bq_ndcg_result.csv"

NAME_MAP = {
    "score_overlap": "overlap",
    "score_tfidf": "TF-IDF (cosine)",
    "score_bm25_default": "BM25 default (k1=1.5, b=0.75)",
    "score_bm25_variant": f"BM25 variant {VARIANT} (k1={K1}, b={B})",
}

if not os.path.exists(BQ_RESULT):
    print(f"Файл {BQ_RESULT} не найден. Сначала выполните запрос B в BigQuery "
          "и сохраните результат через Save results → CSV.")
else:
    bq = pd.read_csv(BQ_RESULT)
    py = comparison.set_index(["method", "k"])["mean_NDCG@k"]
    rows = []
    for _, r in bq.iterrows():
        name = NAME_MAP.get(r["method"], r["method"])
        py_value = float(py.loc[(name, int(r["k"]))])
        rows.append({"method": r["method"], "k": int(r["k"]),
                     "BigQuery": round(float(r["mean_ndcg"]), 4),
                     "Python": round(py_value, 4),
                     "delta": round(float(r["mean_ndcg"]) - py_value, 6)})
    check = pd.DataFrame(rows).sort_values(["k", "BigQuery"], ascending=[True, False])
    display(check)
    worst = check["delta"].abs().max()
    print(f"\nМаксимальное расхождение: {worst:.2e}")
    if worst < 5e-4:
        print("Контроль пройден: SQL и Python дают одинаковый NDCG "
              "(различия — только округление до 4 знаков).")
    else:
        print("Расхождение слишком велико. Проверьте: тот ли CSV загружен; типы "
              "колонок score_* и cand_order; разрешение ничьих ORDER BY ..., "
              "cand_order; условие WHERE idcg > 0.")

### 9.6. Что сдавать по блоку BigQuery

К основному комплекту добавляются:

* `queries.sql` — текст запросов A и B (и C, если выполняли);
* `bq_ndcg_result.csv` — выгруженный результат запроса B;
* скриншот вкладки **Schema** таблицы `wikiqa_scores` (подтверждает, что числовые
  колонки не стали `STRING`);
* в `README.md` — короткий абзац: совпали ли значения NDCG в SQL и Python, и если
  нет, то в чём была причина.

Сам файл `wikiqa_scores.csv` в репозиторий **не коммитится**: он производен от
корпуса WikiQA и подпадает под те же условия лицензии.

**Частые ошибки в этом блоке**

1. Забыто `WHERE idcg > 0` — метрика занижена примерно в 2,6 раза.
2. Не задано разрешение ничьих (`, cand_order` в `ORDER BY`). У метода `overlap`
   оценки целочисленные, ничьих много, и без этого NDCG будет меняться от запуска
   к запуску.
3. `Auto detect` определил `score_*` как `STRING` — сортировка пойдёт по тексту,
   и «9.5» окажется больше «10.0».
4. Использован `LOG(pos + 1)` вместо `LOG(pos + 1, 2)` — натуральный логарифм
   вместо двоичного, все значения пропорционально смещены.
5. Взята отсечка `pos <= k` для DCG, но не для IDCG — знаменатель посчитан по
   полной выдаче, NDCG занижен на запросах с несколькими ответами.

## 11. Повышенная часть (по желанию, до +1 балла)

Достаточно **одного** из пунктов.

**А. PostgreSQL.** Загрузите кандидатов в таблицу, постройте `tsvector`/GIN-индекс,
отранжируйте кандидатов одного вопроса через `ts_rank_cd(to_tsvector(...),
plainto_tsquery(...))` и сравните выдачу с Python-реализацией.
Внимание: `ts_rank` — **не** BM25, в нём нет ни $k_1$, ни $b$; задача — сравнить
встроенное ранжирование СУБД с вашим кодом, а не объявить формулы одинаковыми.

**Б. Предобработка.** Измерьте NDCG@k при четырёх режимах: со стоп-словами и без,
со стеммингом и без (`make_tokenizer(remove_stopwords=..., stem=...)`, для этого
понадобится доработать `tokenize` под параметр `stem`). Сравните размах результатов
с влиянием параметров $k_1, b$.

**В. Гибрид.** Объедините TF-IDF и BM25 через Reciprocal Rank Fusion
$\mathrm{RRF}(d)=\sum_m \frac{1}{60+\mathrm{rank}_m(d)}$ и проверьте, даёт ли гибрид
прирост. Отрицательный результат тоже засчитывается, если он измерен и объяснён.

**Г. Answer triggering.** 62 % вопросов WikiQA вообще не имеют ответа среди кандидатов.
Попробуйте по максимальной оценке BM25 предсказать, есть ли ответ (метрика — ROC-AUC
по всем вопросам) и объясните, почему эту задачу нельзя смешивать с ранжированием.

In [ ]:
# TODO (по желанию): повышенная часть

## 12. Полная самопроверка перед сдачей

In [ ]:
# ВЫДАНО: финальная самопроверка
def check_leakage():
    assert check_no_label_leakage(tfidf_scorer, ELIGIBLE), "TF-IDF использует Label"
    assert check_no_label_leakage(bm25_variant_scorer, ELIGIBLE), "BM25 использует Label"


def check_table():
    assert comparison is not None and len(comparison) > 0, \
        "сводная таблица (задание 6) не построена"


def check_source():
    assert DATA_SOURCE != "synthetic", \
        "источник 'synthetic': загрузите настоящий WikiQA-test.tsv"


def check_analysis():
    assert pq_tfidf is not None and pq_bm25 is not None, \
        "per-query анализ (задание 7) не выполнен"


def final_check():
    checks = []
    for name, fn in [("токенизация", test_tokenize),
                     ("TF-IDF", test_tfidf),
                     ("BM25 и параметры варианта", test_bm25),
                     ("DCG/NDCG", test_metrics),
                     ("нет утечки Label", check_leakage),
                     ("сводная таблица", check_table),
                     ("per-query анализ", check_analysis),
                     ("данные — настоящий WikiQA", check_source)]:
        try:
            fn()
            checks.append((name, True, ""))
        except Exception as exc:                      # noqa: BLE001
            checks.append((name, False, f"{type(exc).__name__}: {exc}"))

    print()
    width = max(len(name) for name, _, _ in checks)
    for name, ok, msg in checks:
        print(f"{'[OK]  ' if ok else '[FAIL]'} {name.ljust(width)}  {msg}")
    failed = [name for name, ok, _ in checks if not ok]
    print("\n" + ("Всё готово к сдаче." if not failed
                  else f"Не пройдено: {', '.join(failed)}"))

final_check()

> **Если выполнялся блок BigQuery (задание 9)**, перед сдачей убедитесь
> дополнительно: `test_export()` проходит, файл `bq_ndcg_result.csv` получен,
> а ячейка сверки напечатала «Контроль пройден».

### Что сдавать

Репозиторий (или отдельная ветка) со структурой:

```
lab_02/
├── README.md            # ФИО, группа, вариант, метод, результат, вывод
├── lab_02.ipynb         # этот ноутбук, выполненный сверху вниз
├── requirements.txt
└── solution.py          # при необходимости
```

Сам корпус WikiQA в репозиторий **не коммитится** — только код его загрузки
(условия Microsoft Research Data License). Ноутбук должен воспроизводиться после
`pip install -r requirements.txt`.